# Build the semantic (sbert) retrieval index on a free GPU

Encodes the 77,348 label chunks with `BAAI/bge-small-en-v1.5` (a pretrained sentence-embedding model, used as is, **not fine-tuned**) and writes
`sbert_index.zip`. Runs on **Kaggle**. The chunks are never rebuilt here: the notebook uses the `chunks.jsonl` you upload, and it stops if its
hash does not match the manifest.

**Before you start**
1. Push the commit that contains this notebook and the `--device` option of `scripts/build_index.py` to the GitHub branch this notebook clones (`BRANCH` below, `main`). **The notebook clones `main`, so it fails (or runs old code) until that push is done.**
2. Kaggle: create a *private dataset* named `medsafe-index` and upload `aakaash/artifacts/index/chunks.jsonl` (about 215 MB) and `aakaash/artifacts/index/manifest.json` (a few KB). Nothing else is needed. Add the dataset to this notebook.
3. Settings > Accelerator > a GPU (T4 or P100); Internet on (to clone the repo and download the model).

**Time (an estimate until measured):** minutes to about an hour on a T4, against about 6 hours on the local CPU. The encoding cell prints its own timing.
Nothing here is committed to git.

In [ ]:
!nvidia-smi

In [ ]:
# Clear /kaggle/working first so an old run cannot leak into this one, then find the uploaded index files.
import os, glob, json, pathlib, shutil, subprocess, sys, hashlib
ON_KAGGLE = os.path.exists('/kaggle/working')
assert ON_KAGGLE, 'This notebook is written for Kaggle (/kaggle/working not found).'
WORK = pathlib.Path('/kaggle/working')
for p in WORK.iterdir():
    shutil.rmtree(p) if p.is_dir() else p.unlink()
REPO_URL = 'https://github.com/milanjohnson/Medication_Safety_and_Decision_support_system'
BRANCH = 'main'

chunks_hits = sorted(glob.glob('/kaggle/input/**/chunks.jsonl', recursive=True))
assert chunks_hits, 'chunks.jsonl not found under /kaggle/input: attach the dataset medsafe-index (see the first cell)'
tops = []
for h in sorted(glob.glob('/kaggle/input/**/manifest.json', recursive=True)):
    try:
        j = json.load(open(h, encoding='utf-8'))
    except Exception:
        continue
    if 'chunks_sha256' in j and 'sections' in j and 'labels' in j:   # the top-level index manifest (not the tfidf / sbert ones)
        tops.append(pathlib.Path(h))
print('chunks.jsonl candidates:', chunks_hits)
print('top manifest candidates:', [str(t) for t in tops])
assert len(chunks_hits) == 1, 'more than one chunks.jsonl found; attach only the medsafe-index dataset'
assert len(tops) == 1, 'expected exactly one top-level index manifest.json (the one next to chunks.jsonl)'
CHUNKS_SRC, MANIFEST_SRC = pathlib.Path(chunks_hits[0]), tops[0]

In [ ]:
%cd {WORK}
!git clone --depth 1 -b {BRANCH} {REPO_URL} repo
%cd {WORK}/repo/aakaash
!git log --oneline -1
# The code this notebook needs must be on the branch: fail now, not after the install.
assert '--device' in open('scripts/build_index.py', encoding='utf-8').read(), 'build_index.py on this branch has no --device option: push the new commit to main first'
# Do not change numpy inside a running session (that corrupts it). Keep the numpy this image already has and install the rest.
import numpy
NP = numpy.__version__
!grep -v '^numpy' requirements-train.txt > /tmp/req_train.txt
!pip install -q -r /tmp/req_train.txt "numpy=={NP}"
print('numpy kept at', NP)
import torch
print('torch', torch.__version__, 'cuda', torch.version.cuda, 'available', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU: Settings > Accelerator > GPU, then restart the session'
GPU = torch.cuda.get_device_name(0)
print('GPU:', GPU)

In [ ]:
# Copy the two files into a working index folder and check the hash of the chunks against the manifest before spending GPU time.
IDX = WORK / 'repo/aakaash/artifacts/index'
IDX.mkdir(parents=True, exist_ok=True)
shutil.copy(CHUNKS_SRC, IDX / 'chunks.jsonl')
shutil.copy(MANIFEST_SRC, IDX / 'manifest.json')
top = json.load(open(IDX / 'manifest.json', encoding='utf-8'))
INPUT_SHA = top['chunks_sha256']
h = hashlib.sha256()
with open(IDX / 'chunks.jsonl', 'rb') as fh:
    for block in iter(lambda: fh.read(1 << 24), b''):
        h.update(block)
print('manifest chunks', top['chunks'], 'labels', top['labels'])
print('manifest sha256', INPUT_SHA)
print('file     sha256', h.hexdigest())
assert h.hexdigest() == INPUT_SHA, 'chunks.jsonl does not match its manifest: re-upload both files from the same build'
assert top['chunks'] == 77348, f"expected 77,348 chunks, the manifest says {top['chunks']}"

In [ ]:
# Encode on the GPU. Resumable: if the session stops, run this cell again (finished shards are kept).
import time
t0 = time.time()
!ENCODER=sbert python scripts/build_index.py --out {IDX} --device cuda
ELAPSED = time.time() - t0
print(f'build_index.py finished in {ELAPSED/60:.1f} min')

In [ ]:
# Verify. Fails loudly if the hash differs or the GPU was not used.
out = json.load(open(IDX / 'sbert/manifest.json', encoding='utf-8'))
assert out['chunks_sha256'] == INPUT_SHA, f"HASH MISMATCH: input {INPUT_SHA} output {out['chunks_sha256']}"
assert out['chunks'] == top['chunks']
assert out.get('device') == 'cuda' and out.get('gpu'), f"the GPU was not used (device={out.get('device')})"
import numpy as np
V = np.load(IDX / 'sbert/vectors.npy', mmap_mode='r'); owner = np.load(IDX / 'sbert/window_chunk.npy')
assert len(V) == len(owner) == out['windows'] and V.shape[1] == out['dimension']
assert int(owner.max()) == top['chunks'] - 1 and len(np.unique(owner)) == top['chunks'], 'some chunk has no window'
assert np.isfinite(np.asarray(V[:2000])).all()
print('hash matches the input manifest')
print('GPU            :', out['gpu'], '| batch size', out['batch_size'])
print('model          :', out['model'], 'revision', out['model_revision'])
print('timing         :', out['seconds'], 's in build_sbert,', round(ELAPSED), 's including start-up')
print('chunks         :', out['chunks'])
print('windows        :', out['windows'], f"({out['chunks_with_more_than_one_window']} chunks have more than one window)")
print('dimension      :', out['dimension'])
print('vector bytes   :', out['vector_bytes'])

In [ ]:
# Zip ONLY vectors.npy, window_chunk.npy and manifest.json (no shards). This is the file to download.
import zipfile
ZIP = WORK / 'sbert_index.zip'
with zipfile.ZipFile(ZIP, 'w', zipfile.ZIP_DEFLATED) as z:
    for name in ('vectors.npy', 'window_chunk.npy', 'manifest.json'):
        z.write(IDX / 'sbert' / name, name)
print(zipfile.ZipFile(ZIP).namelist(), round(ZIP.stat().st_size / 1e6, 1), 'MB')

## What to download and where it goes

Download `sbert_index.zip` from the Output panel (`/kaggle/working/sbert_index.zip`). On your computer, from the `aakaash` folder:

```
.venv\Scripts\python.exe scripts\install_sbert.py C:\path\to\sbert_index.zip --force
.venv\Scripts\python.exe scripts\compare_encoders.py
```

`--force` is needed only because an unfinished `artifacts/index/sbert/shards` folder from the earlier CPU attempt exists. The installer refuses a zip whose `chunks_sha256` differs from `artifacts/index/manifest.json`. Nothing is committed to git.